# MAX-3 · Maximum Model — Single-Fold Test (OAI)

Trains the maximum configuration on the OAI fold: CORN ordinal head, full
backbone fine-tuning, 384×384 resolution, soft ordinal targets, source-reliability
weighting, sampling, curriculum, and domain-adversarial training. Results are
written to scope3_max; the original pipeline is untouched. This single fold
confirms whether the maximum stack improves on the baseline before running all
four folds.

## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np, torch
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU (A100).')
print('GPU:', torch.cuda.get_device_name(0))
print('Resolution:', TM.MAX_IMG_SIZE, '| Freeze:', TM.MAX_FREEZE_STAGES, '| MRKR trust:', TM.MRKR_TRUST)
manifest = TM.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Resolution: 384 | Freeze: 0 | MRKR trust: 0.4
Copied array in 48s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


## Train maximum model on Mendeley fold

In [3]:
mech = {'sampler':True,'noise_loss':True,'curriculum':True,'domain_adv':True,
        'hierarchical':True,'use_quality':True,'soft_alpha':TM.MAX_SOFT_ALPHA,
        'freeze_stages':TM.MAX_FREEZE_STAGES,'grl_lambda_max':0.5}

test_ds = 'oai'
tr, va, te = TM.make_splits(manifest, test_ds, seed=0)
print(f'train={len(tr):,}  val={len(va):,}  test={len(te):,}')

res = TM.run_training('max_oai_seed0', tr, va, te, mech, seed=0,
                      log_fn=print)

train=45,059  val=7,952  test=8,547
  [max_oai_seed0] complete - skip.


## Compare to baseline (original H, Mendeley fold)

In [4]:
import json, glob
def load(p):
    try: return json.load(open(str(p)))
    except Exception: return None

h = load(config.RESULTS_DIR/'fold4_test_oai_seed0.json')
h_w1 = np.nan
zf = sorted(glob.glob(str(config.RESULTS_DIR/'fold4_test_oai_seed*_preds.npz')))
if zf:
    z=np.load(zf[0]); h_w1=float((np.abs(z['y_true']-z['y_pred'])<=1).mean())

print('OAI fold — baseline H vs MAX (seed 0):')
print(f"{'metric':12s}{'H (224)':>12s}{'MAX (384)':>12s}")
if h:
    print(f"{'exact acc':12s}{h['external_acc5']:>12.3f}{res['external_acc5']:>12.3f}")
    print(f"{'within-1':12s}{h_w1:>12.3f}{res['external_within1']:>12.3f}")
    print(f"{'QWK':12s}{h['external_qwk']:>12.3f}{res['external_qwk']:>12.3f}")
    print(f"{'gap (pp)':12s}{h['gap']*100:>12.1f}{res['gap']*100:>12.1f}")
    print()
    print(f"Delta exact: {res['external_acc5']-h['external_acc5']:+.3f}")
    print(f"Delta within-1: {res['external_within1']-h_w1:+.3f}")
else:
    print(f"MAX: exact={res['external_acc5']:.3f} within1={res['external_within1']:.3f} qwk={res['external_qwk']:.3f} gap={res['gap']*100:.1f}pp")

OAI fold — baseline H vs MAX (seed 0):
metric           H (224)   MAX (384)
exact acc          0.484       0.522
within-1           0.782       0.807
QWK                0.533       0.577
gap (pp)             4.0         3.5

Delta exact: +0.038
Delta within-1: +0.025
